# Notebook 5 — Selective Retraining Simulation

**Purpose:** Run the full retraining simulation experiment across all dataset,
model, strategy, and trigger combinations. This is the core experimental notebook
of the thesis, producing the results analysed in Chapter 4.

**Experimental design:**
- **Datasets:** 3 (Electricity real, Gradual concept drift, Abrupt concept drift)
- **Models:** 2 (Logistic Regression, XGBoost)
- **Strategies:** 5 (No retrain, Continuous, Cumulative, Sliding Window, Exponential Decay)
- **Label availability:** 2 (label-available → performance trigger, label-delayed → PSI trigger)
- **Total runs:** 60 (3 × 2 × 5 × 2)
- **Random seed:** 42 (single seed; deterministic results)

**Key configuration parameters:**
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `BATCH_SIZE` | 500 | Matches synthetic dataset batch structure |
| `INITIAL_TRAIN_RATIO` | 0.20 | First 20% used for baseline training |
| `THRESHOLD_RELATIVE` | 0.95 | Retrain if accuracy < 95% of baseline |
| `THRESHOLD_MINIMUM` | 0.90 | Hard floor for synthetic datasets |
| `THRESHOLD_MINIMUM_ELECTRICITY` | 0.75 | Hard floor for real dataset (lower baseline) |
| `WINDOW_SIZE` | 5 | Sliding window: last 5 batches |
| `DECAY_ALPHA` | 0.9 | Exponential decay factor (w = 0.9^age) |
| `PSI_THRESHOLD` | 0.25 | PSI trigger threshold (significant drift) |

**Outputs:**
- `results_overall.csv` / `.xlsx` — one row per simulation run (60 rows)
- `results_batchwise.csv` / `.xlsx` — one row per batch per run (60 × 72 = 4,320 rows)
- `artifact_index.csv` — index of all generated figures and tables

---
| Cell | Content |
|------|---------|
| 1 | Setup & package installation |
| 2 | Global configuration |
| 2.5 | Hyperparameter tuning (runs once per dataset) |
| 3 | Mount Google Drive |
| 4 | Synthetic dataset generation |
| 5 | Load all datasets into memory |
| 6 | Core utilities (PSI, model training, evaluation) |
| 7 | Simulation engine (`run_simulation`) |
| 8 | Simulation matrix (60-run specification) |
| 9 | Execute all 60 simulations |
| 10 | Export results to CSV/Excel |
| 11 | Quick sanity check on exported results |
| 12 | Descriptive pairwise strategy comparison |

---
## Cell 1 — Setup & Package Installation

All required packages are installed here. This cell should be run once
at the start of each Colab session.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install xgboost scikit-learn pandas numpy openpyxl -q

import numpy as np
import pandas as pd
import os
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ All libraries loaded.")

✅ All libraries loaded.


---
## Cell 2 — Global Configuration

All experiment parameters are defined in a single configuration block.
To reproduce this experiment exactly, keep all values as shown.
Changes to `BATCH_SIZE`, `DECAY_ALPHA`, or threshold values will
alter the simulation results.

In [3]:
# ── GLOBAL CONFIGURATION ─────────────────────────────────────────
# Edit DATA_PATH before running.

DATA_PATH   = "/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets"
OUTPUT_PATH = DATA_PATH + "/results"

FEATURES            = ['nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer']
TARGET              = 'class'

BATCH_SIZE          = 500
INITIAL_TRAIN_RATIO = 0.20
TRAIN_VAL_SPLIT     = 0.80

THRESHOLD_RELATIVE  = 0.95
THRESHOLD_MINIMUM             = 0.90   # synthetic datasets
THRESHOLD_MINIMUM_ELECTRICITY = 0.75   # electricity (real dataset, harder baseline)
PSI_THRESHOLD       = 0.25

WINDOW_SIZE         = 5
DECAY_ALPHA         = 0.9
PSI_BINS            = 10
PSI_CAP             = 1.0

# ── Multi-seed configuration ─────────────────────────────────────
SEEDS = [42]  #[42, 123, 456, 789, 2024] # 5 seeds — change count here

print("✅ Configuration set.")
print(f"   DATA_PATH   : {DATA_PATH}")
print(f"   OUTPUT_PATH : {OUTPUT_PATH}")

✅ Configuration set.
   DATA_PATH   : /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets
   OUTPUT_PATH : /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results


---
## Cell 2.5 — Hyperparameter Tuning

Optimises Logistic Regression and XGBoost hyperparameters independently
for each dataset using 5-fold cross-validation on the initial training window.

**This cell runs once and saves results to `best_params.json`.**
On subsequent runs, load the saved file instead of re-running tuning.
The optimised parameters are used in all 60 simulation runs via `build_model()`.

> Hyperparameter tuning is performed on the baseline training window only,
> not on streaming data, to avoid data leakage from future batches.

In [4]:
import json
from sklearn.model_selection import GridSearchCV, StratifiedKFold

PARAMS_PATH = os.path.join(DATA_PATH, "best_params.json")

DATASET_FILES = {
    "electricity"                : "electricity_clean.xlsx",
    "electricity_gradual_concept": "gradual_concept.xlsx",
    "electricity_abrupt_concept" : "abrupt_concept.xlsx",
}

LR_GRID = {
    "C"       : [0.01, 0.1, 1.0, 10.0],
    "solver"  : ["lbfgs", "liblinear"],
    "max_iter": [500],
}

XGB_GRID = {
    "n_estimators" : [100, 200],
    "max_depth"    : [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample"    : [0.8, 1.0],
}

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def tune_dataset(df, dataset_name):
    """
    Tune LR and XGBoost on the initial training split of df.
    Returns dict with best params for each model, plus validation accuracy.
    """
    # Replicate exact split used in simulation
    all_batches = sorted(df["batch_id"].unique())
    n_init      = int(len(all_batches) * INITIAL_TRAIN_RATIO)
    init_ids    = all_batches[:n_init]
    init_data   = df[df["batch_id"].isin(init_ids)].copy()

    n       = len(init_data)
    n_train = int(n * TRAIN_VAL_SPLIT)
    train   = init_data.iloc[:n_train]
    val     = init_data.iloc[n_train:]

    X_train = train[FEATURES].values
    y_train = train[TARGET].values
    X_val   = val[FEATURES].values
    y_val   = val[TARGET].values

    result = {}

    # ── Logistic Regression ──────────────────────────────────
    print(f"  [LR]  Tuning on {dataset_name}...")
    lr_search = GridSearchCV(
        LogisticRegression(random_state=42),
        LR_GRID, cv=CV, scoring="accuracy", n_jobs=-1, refit=True
    )
    lr_search.fit(X_train, y_train)
    lr_val_acc = accuracy_score(y_val, lr_search.best_estimator_.predict(X_val))
    result["logistic_regression"] = {
        "params"   : lr_search.best_params_,
        "cv_acc"   : round(lr_search.best_score_, 4),
        "val_acc"  : round(lr_val_acc, 4),
    }
    _min_lr = 0.75 if dataset_name == "electricity" else 0.80
    status = "✅" if lr_val_acc >= _min_lr else f"⚠️  BELOW {_min_lr}"
    print(f"        best={lr_search.best_params_}  val_acc={lr_val_acc:.4f}  {status}")

    # ── XGBoost ──────────────────────────────────────────────
    print(f"  [XGB] Tuning on {dataset_name}...")
    xgb_search = GridSearchCV(
        XGBClassifier(
            use_label_encoder=False, eval_metric="logloss",
            random_state=42, verbosity=0
        ),
        XGB_GRID, cv=CV, scoring="accuracy", n_jobs=-1, refit=True
    )
    xgb_search.fit(X_train, y_train)
    xgb_val_acc = accuracy_score(y_val, xgb_search.best_estimator_.predict(X_val))
    result["xgboost"] = {
        "params"  : xgb_search.best_params_,
        "cv_acc"  : round(xgb_search.best_score_, 4),
        "val_acc" : round(xgb_val_acc, 4),
    }
    _min_xgb = 0.75 if dataset_name == "electricity" else 0.80
    status = "✅" if xgb_val_acc >= _min_xgb else f"⚠️  BELOW {_min_xgb}"
    print(f"        best={xgb_search.best_params_}  val_acc={xgb_val_acc:.4f}  {status}")

    return result


# ── Main tuning routine ──────────────────────────────────────
if os.path.exists(PARAMS_PATH):
    print(f"⏭️  best_params.json already exists — skipping tuning.")
    print(f"   Path: {PARAMS_PATH}")
    with open(PARAMS_PATH) as f:
        best_params = json.load(f)
    print("\n📋 Loaded params summary:")
    for ds, models in best_params.items():
        for m, info in models.items():
            print(f"   {ds:35s} | {m:22s} | val_acc={info['val_acc']:.4f} | {info['params']}")
else:
    print("🔧 Starting hyperparameter tuning (once per dataset)...\n")
    best_params = {}

    for dataset_name, filename in DATASET_FILES.items():
        fpath = os.path.join(DATA_PATH, filename)
        print(f"\n📂 Dataset: {dataset_name}")
        df_tmp = pd.read_excel(fpath)
        if "batch_id" not in df_tmp.columns:
            df_tmp["batch_id"] = np.arange(len(df_tmp)) // BATCH_SIZE
        best_params[dataset_name] = tune_dataset(df_tmp, dataset_name)

    with open(PARAMS_PATH, "w") as f:
        json.dump(best_params, f, indent=2)

    print(f"\n✅ Tuning complete. Saved → {PARAMS_PATH}")
    print("\n📋 Final params summary:")
    for ds, models in best_params.items():
        for m, info in models.items():
            print(f"   {ds:35s} | {m:22s} | val_acc={info['val_acc']:.4f} | {info['params']}")


⏭️  best_params.json already exists — skipping tuning.
   Path: /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/best_params.json

📋 Loaded params summary:
   electricity                         | logistic_regression    | val_acc=0.7711 | {'C': 10.0, 'max_iter': 500, 'solver': 'lbfgs'}
   electricity                         | xgboost                | val_acc=0.7706 | {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
   electricity_gradual_concept         | logistic_regression    | val_acc=0.9567 | {'C': 10.0, 'max_iter': 500, 'solver': 'lbfgs'}
   electricity_gradual_concept         | xgboost                | val_acc=0.9567 | {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
   electricity_abrupt_concept          | logistic_regression    | val_acc=0.9567 | {'C': 10.0, 'max_iter': 500, 'solver': 'lbfgs'}
   electricity_abrupt_concept  

---
## Cell 3 — Mount Google Drive

Mounts the Drive and sets the data path. All datasets and output files
are read from and written to `DATA_PATH`.

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')
# os.makedirs(OUTPUT_PATH, exist_ok=True)
# print("✅ Drive mounted and output directory ready.")

---
## Cell 4 — Synthetic Datasets

> Synthetic dataset generation has been moved to **Notebook 3**
> (`2 synthetic dataset generation.ipynb`).
>
> Before running this notebook, ensure that the following files
> exist in `ready_to_use_datasets/`:
> - `electricity_clean.xlsx` (from Notebook 1)
> - `gradual_concept.xlsx` (from Notebook 3)
> - `abrupt_concept.xlsx` (from Notebook 3)

The cell below verifies all required files are present before proceeding.

In [6]:
import os

# ── Verify all required dataset files exist ──────────────────────────
REQUIRED = [
    os.path.join(DATA_PATH, 'electricity_clean.xlsx'),
    os.path.join(DATA_PATH, 'gradual_concept.xlsx'),
    os.path.join(DATA_PATH, 'abrupt_concept.xlsx'),
]

all_present = True
for fpath in REQUIRED:
    exists = os.path.exists(fpath)
    status = '✓' if exists else '✗  MISSING'
    print(f'  {status}  {os.path.basename(fpath)}')
    if not exists:
        all_present = False

if not all_present:
    raise FileNotFoundError(
        'One or more dataset files are missing. '
        'Run Notebook 1 (electricity preparation) and '
        'Notebook 3 (synthetic dataset generation) first.'
    )

print('\n✅ All required dataset files found. Ready to proceed.')


  ✓  electricity_clean.xlsx
  ✓  gradual_concept.xlsx
  ✓  abrupt_concept.xlsx

✅ All required dataset files found. Ready to proceed.


---
## Cell 5 — Load All Datasets

Loads all three experimental datasets into memory as pandas DataFrames.
Each dataset is verified to contain the expected columns and row count
before the simulation begins.

In [7]:
datasets = {
    'electricity':                 pd.read_excel(os.path.join(DATA_PATH, 'electricity_clean.xlsx')),
    'electricity_gradual_concept': pd.read_excel(os.path.join(DATA_PATH, 'gradual_concept.xlsx')),
    'electricity_abrupt_concept':  pd.read_excel(os.path.join(DATA_PATH, 'abrupt_concept.xlsx')),
}

for name, df in datasets.items():
    print(f"  {name:40s} → {df.shape[0]:>6} rows, {df.shape[1]:>2} cols")

print("\n✅ All datasets loaded.")

  electricity                              →  45000 rows, 10 cols
  electricity_gradual_concept              →  45000 rows, 10 cols
  electricity_abrupt_concept               →  45000 rows, 10 cols

✅ All datasets loaded.


---
## Cell 6 — Core Utilities

Defines all helper functions used by the simulation engine:
- `calculate_psi()` — mean PSI across features vs reference distribution
- `build_model()` — instantiates LR or XGB with tuned hyperparameters
- `train_model()` — fits model with optional `sample_weight` (used by exponential decay)
- `evaluate_batch()` — returns accuracy, AUC, F1, precision, recall for one batch
- `compute_decay_weights()` — calculates per-sample weights: `w = alpha^(current_batch - sample_batch)`

In [8]:
# ── PSI CALCULATION ─────────────────────────────────────────────
# Per-feature PSI, capped at PSI_CAP, averaged across features.
# Reference updates after each retraining event.

def calculate_psi(reference, actual):
    psi_values = []
    for col in FEATURES:
        e = reference[col].values
        a = actual[col].values
        breakpoints = np.unique(np.percentile(e, np.linspace(0, 100, PSI_BINS + 1)))
        if len(breakpoints) < 3:
            psi_values.append(0.0)
            continue
        e_counts = np.histogram(e, bins=breakpoints)[0] / len(e)
        a_counts = np.histogram(a, bins=breakpoints)[0] / len(a)
        e_counts = np.clip(e_counts, 1e-4, None)
        a_counts = np.clip(a_counts, 1e-4, None)
        psi = float(np.sum((e_counts - a_counts) * np.log(e_counts / a_counts)))
        psi_values.append(min(psi, PSI_CAP))
    return float(np.mean(psi_values))


# ── MODEL FACTORY ────────────────────────────────────────────────

def build_model(model_type, dataset_name, seed=42):
    """Build model using tuned hyperparameters from best_params.json."""
    params = best_params.get(dataset_name, {}).get(model_type, {}).get('params', {})
    if model_type == 'logistic_regression':
        return LogisticRegression(
            max_iter=params.get('max_iter', 500),
            C=params.get('C', 1.0),
            solver=params.get('solver', 'lbfgs'),
            random_state=seed
        )
    return XGBClassifier(
        n_estimators=params.get('n_estimators', 100),
        max_depth=params.get('max_depth', 4),
        learning_rate=params.get('learning_rate', 0.1),
        subsample=params.get('subsample', 1.0),
        use_label_encoder=False, eval_metric='logloss',
        random_state=seed, verbosity=0
    )


class ConstantBinaryClassifier:
    """Fallback model for retraining windows that contain only one class."""
    def __init__(self, constant_class):
        self.constant_class = int(constant_class)
        self.classes_ = np.array([0, 1])

    def predict(self, X):
        return np.full(len(X), self.constant_class, dtype=int)

    def predict_proba(self, X):
        proba = np.zeros((len(X), 2), dtype=float)
        proba[:, self.constant_class] = 1.0
        return proba


def train_model(data, model_type, dataset_name, sample_weight=None, seed=42, use_validation=True):
    """Train model. Initial training uses validation; retraining can fit on all selected data."""
    X = data[FEATURES].values
    y = data[TARGET].values.astype(int)

    unique_classes, class_counts = np.unique(y, return_counts=True)
    if len(unique_classes) < 2:
        model = ConstantBinaryClassifier(unique_classes[0])
        val_acc = accuracy_score(y, model.predict(X)) if use_validation else None
        return model, val_acc

    if not use_validation:
        model = build_model(model_type, dataset_name, seed=seed)
        model.fit(X, y, sample_weight=sample_weight)
        return model, None

    stratify_labels = y if np.min(class_counts) >= 2 else None

    idx = np.arange(len(X))
    idx_train, idx_val = train_test_split(
        idx,
        test_size=(1 - TRAIN_VAL_SPLIT),
        random_state=seed,
        stratify=stratify_labels
    )
    X_train, X_val = X[idx_train], X[idx_val]
    y_train, y_val = y[idx_train], y[idx_val]

    # Rare small/skewed windows can lose a class after splitting; fall back instead of crashing.
    if len(np.unique(y_train)) < 2:
        model = ConstantBinaryClassifier(y_train[0])
        val_acc = accuracy_score(y_val, model.predict(X_val))
        return model, val_acc

    model  = build_model(model_type, dataset_name, seed=seed)
    w_train = sample_weight[idx_train] if sample_weight is not None else None
    model.fit(X_train, y_train, sample_weight=w_train)
    val_acc = accuracy_score(y_val, model.predict(X_val))
    return model, val_acc


def evaluate_batch(model, batch_df):
    """Return (accuracy, auc, f1, precision, recall) for a single batch."""
    X    = batch_df[FEATURES].values
    y    = batch_df[TARGET].values
    pred = model.predict(X)
    acc  = accuracy_score(y, pred)
    f1   = f1_score(y, pred, zero_division=0)
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    try:
        auc = roc_auc_score(y, model.predict_proba(X)[:, 1])
    except Exception:
        auc = np.nan
    return acc, auc, f1, prec, rec


# ── EXPONENTIAL DECAY WEIGHTS ────────────────────────────────────
# All samples in the same batch receive the same weight.
# Weight = alpha ^ (current_batch_id - sample_batch_id)

def compute_decay_weights(batch_id_array, current_b):
    ages = current_b - np.array(batch_id_array)
    return (DECAY_ALPHA ** ages).astype(float)


print("✅ Core utilities defined.")

✅ Core utilities defined.


---
## Cell 7 — Simulation Engine

`run_simulation()` executes one complete streaming simulation run.
It processes all 72 streaming batches sequentially, applying the
specified retraining strategy and trigger mechanism.

**For each streaming batch, the engine:**
1. Evaluates the current model (accuracy, AUC, F1, precision, recall)
2. Computes PSI against the current reference distribution
3. Checks the trigger condition (performance drop or PSI threshold)
4. If triggered: updates training data, reweights samples (exponential decay), retrains model
5. Tracks recovery time, recovery gain, and retraining cost
6. Records all metrics in `batchwise_records` (72 rows) and `overall_record` (1 row)

**Returns:** `(batchwise_records, overall_record)`

In [9]:
def run_simulation(df_input, dataset_name, model_type, strategy, label_available, sim_no, seed=42):
    """
    Runs one simulation end-to-end.

    Parameters
    ----------
    df_input        : full dataset DataFrame with 'batch_id' column
    dataset_name    : string identifier
    model_type      : 'logistic_regression' | 'xgboost'
    strategy        : 'no_retraining' | 'continuous' | 'cumulative'
                      | 'sliding_window' | 'exponential_decay'
    label_available : True  → dual-baseline trigger (performance)
                      False → PSI trigger
    sim_no          : simulation number 1-60
    seed            : random seed for reproducibility

    Returns
    -------
    batchwise_records : list of dicts (one per streaming batch)
    overall_record    : dict (one aggregated record)
    """
    np.random.seed(seed)

    # 1. SPLIT INITIAL VS STREAMING BATCHES
    all_batches  = sorted(df_input['batch_id'].unique())
    n_initial    = int(len(all_batches) * INITIAL_TRAIN_RATIO)
    initial_ids  = all_batches[:n_initial]
    stream_ids   = all_batches[n_initial:]
    initial_data = df_input[df_input['batch_id'].isin(initial_ids)].copy()

    # 2. INITIAL MODEL TRAINING (80% train / 20% val internal split)
    model, val_acc = train_model(initial_data, model_type, dataset_name, seed=seed)

    # 3. STATE INITIALISATION
    # Dataset-specific minimum threshold
    threshold_min           = 0.75 if dataset_name == "electricity" else THRESHOLD_MINIMUM

    baseline_accuracy       = val_acc
    model_version           = 1
    psi_reference           = initial_data[FEATURES].copy()
    history_df              = initial_data.copy()
    history_batch_ids       = list(initial_data['batch_id'].values)

    retrain_count           = 0
    total_retrain_data      = 0
    cumulative_retrain_data = 0
    batches_since_retrain   = 0

    degradation_active      = False
    degradation_start_env   = None
    recovery_times          = []
    recovery_gains          = []

    batch_accuracies        = []
    batch_aucs              = []
    batch_f1s               = []
    batch_precisions        = []
    batch_recalls           = []
    retrain_data_per_event  = []
    trigger_types           = []
    false_drift_count       = 0
    missed_drift_count      = 0
    below_095_count         = 0
    degraded_count          = 0

    batchwise_records       = []

    # 4. STREAMING LOOP
    for env_id, batch_id in enumerate(stream_ids):

        current_batch = df_input[df_input['batch_id'] == batch_id].copy()

        # Evaluate current model on this batch
        acc, auc, f1, prec, rec = evaluate_batch(model, current_batch)

        # PSI — always computed for logging regardless of trigger type
        psi_mean = calculate_psi(psi_reference, current_batch[FEATURES])

        # Derived performance metrics
        accuracy_relative = acc / baseline_accuracy if baseline_accuracy > 0 else 0.0
        performance_drop  = baseline_accuracy - acc
        positive_rate     = float(current_batch[TARGET].mean())

        # Decision flags
        is_degraded     = (accuracy_relative < THRESHOLD_RELATIVE) or (acc < threshold_min)
        is_below_min    = (acc < threshold_min)
        is_drift        = (psi_mean >= PSI_THRESHOLD)
        is_false_drift  = is_drift and (not is_degraded)
        is_missed_drift = (not is_drift) and is_degraded

        if is_degraded:   degraded_count   += 1
        if accuracy_relative < THRESHOLD_RELATIVE or is_below_min: below_095_count += 1
        if is_false_drift:  false_drift_count  += 1
        if is_missed_drift: missed_drift_count += 1

        # Degradation episode tracking
        if is_degraded and not degradation_active:
            degradation_active  = True
            degradation_start_env = env_id
        degradation_start_flag = is_degraded and (degradation_start_env == env_id)

        # Recovery check
        is_recovery_complete = False
        recovery_time_val    = None
        if degradation_active and acc >= threshold_min:
            is_recovery_complete  = True
            recovery_time_val     = env_id - degradation_start_env
            recovery_times.append(recovery_time_val)
            degradation_active    = False
            degradation_start_env = None

        # ── TRIGGER LOGIC ────────────────────────────────────────
        retrain_triggered    = False
        retrain_trigger_type = None

        if strategy == 'no_retraining':
            retrain_triggered = False

        elif strategy == 'continuous':
            retrain_triggered    = True
            retrain_trigger_type = 'scheduled'

        else:  # selective strategies
            if label_available:
                # Dual-baseline trigger
                if accuracy_relative < THRESHOLD_RELATIVE:
                    retrain_triggered, retrain_trigger_type = True, 'performance_relative'
                if is_below_min:  # overrides if both fire
                    retrain_triggered, retrain_trigger_type = True, 'performance_minimum'
            else:
                # PSI trigger
                if is_drift:
                    retrain_triggered, retrain_trigger_type = True, 'psi'

        # ── RETRAIN EXECUTION ────────────────────────────────────
        recovery_gain_val = None
        retrain_data_used = 0
        train_set         = None

        if retrain_triggered:
            acc_before = acc

            if strategy in ('continuous', 'cumulative'):
                history_df        = pd.concat([history_df, current_batch], ignore_index=True)
                history_batch_ids += list(current_batch['batch_id'].values)
                train_set          = history_df.copy()
                new_model, _       = train_model(train_set, model_type, dataset_name, seed=seed, use_validation=False)

            elif strategy == 'sliding_window':
                window_start = max(0, env_id - WINDOW_SIZE + 1)
                window_ids   = stream_ids[window_start: env_id + 1]
                train_set    = df_input[df_input['batch_id'].isin(window_ids)].copy()
                new_model, _ = train_model(train_set, model_type, dataset_name, seed=seed, use_validation=False)

            elif strategy == 'exponential_decay':
                history_df        = pd.concat([history_df, current_batch], ignore_index=True)
                history_batch_ids += list(current_batch['batch_id'].values)
                train_set          = history_df.copy()
                weights            = compute_decay_weights(np.array(history_batch_ids), batch_id)
                new_model, _       = train_model(train_set, model_type, dataset_name, sample_weight=weights, seed=seed, use_validation=False)

            retrain_data_used       = len(train_set)
            retrain_count          += 1
            total_retrain_data     += retrain_data_used
            cumulative_retrain_data = total_retrain_data
            retrain_data_per_event.append(retrain_data_used)
            if retrain_trigger_type:
                trigger_types.append(retrain_trigger_type)

            # Update PSI reference to new training data
            psi_reference = train_set[FEATURES].copy()
            model          = new_model
            model_version += 1

            # Recovery gain: new model accuracy on current batch vs before
            acc_after         = evaluate_batch(model, current_batch)[0]
            recovery_gain_val = acc_after - acc_before
            recovery_gains.append(recovery_gain_val)

            # Baseline update: use acc_after on current (next-incoming) batch
            # Update only if acc_after >= minimum threshold
            if acc_after >= threshold_min:
                baseline_accuracy = acc_after

            batches_since_retrain = 0
        else:
            batches_since_retrain += 1

        batch_accuracies.append(acc)
        batch_aucs.append(auc)
        batch_f1s.append(f1)
        batch_precisions.append(prec)
        batch_recalls.append(rec)

        # ── BATCHWISE RECORD ─────────────────────────────────────
        batchwise_records.append({
            # experiment identification
            'sim_no':                     sim_no,
            'env_id':                     env_id,
            'dataset_name':               dataset_name,
            'model_type':                 model_type,
            'memory_strategy':            strategy,
            'label_available':            label_available,
            'seed':                       seed,
            # context
            'model_version':              model_version,
            # data characteristics
            'n_samples':                  len(current_batch),
            'positive_rate':              round(positive_rate, 6),
            # drift metrics
            'psi_mean':                   round(psi_mean, 6),
            # performance metrics
            'accuracy':                   round(acc, 6),
            'auc':                        round(auc, 6) if not np.isnan(auc) else None,
            'f1':                         round(f1, 6),
            'precision':                  round(prec, 6),
            'recall':                     round(rec, 6),
            'baseline_accuracy':          round(baseline_accuracy, 6),
            'accuracy_relative':          round(accuracy_relative, 6),
            'performance_drop':           round(performance_drop, 6),
            # recovery metrics
            'recovery_time':              recovery_time_val,
            'recovery_gain':              round(recovery_gain_val, 6) if recovery_gain_val is not None else None,
            # retraining events
            'retrain_triggered':          retrain_triggered,
            'retrain_trigger_type':       retrain_trigger_type,
            'retrain_count_so_far':       retrain_count,
            'retrain_data':               retrain_data_used,
            'cumulative_retrain_data':    cumulative_retrain_data,
            # temporal robustness
            'batches_since_last_retrain': batches_since_retrain,
            # threshold configuration
            'threshold_relative':         THRESHOLD_RELATIVE,
            'threshold_minimum':          threshold_min,
            'psi_threshold':              PSI_THRESHOLD,
            # threshold status flags
            'is_degraded':                is_degraded,
            'is_below_minimum_threshold': is_below_min,
            'is_drift':                   is_drift,
            'is_false_drift':             is_false_drift,
            'is_missed_drift':            is_missed_drift,
            'degradation_start_flag':     degradation_start_flag,
            'is_recovery_complete':       is_recovery_complete,
        })

    # 5. OVERALL METRICS
    n_stream           = len(stream_ids)
    avg_acc            = float(np.mean(batch_accuracies))
    retrain_count_safe = max(retrain_count, 1)
    total_triggers     = max(len(trigger_types), 1)
    perf_trigs  = sum(1 for t in trigger_types if t in ('performance_relative', 'performance_minimum'))
    psi_trigs   = sum(1 for t in trigger_types if t == 'psi')

    overall_record = {
        'sim_no':                        sim_no,
        'dataset':                       dataset_name,
        'model':                         model_type,
        'strategy':                      strategy,
        'label_available':               label_available,
        'seed':                            seed,
        # classification performance
        'avg_batch_accuracy':            round(avg_acc, 6),
        'avg_batch_auc':                 round(float(np.nanmean(batch_aucs)), 6),
        'avg_batch_f1':                  round(float(np.mean(batch_f1s)), 6),
        'avg_batch_precision':           round(float(np.mean(batch_precisions)), 6),
        'avg_batch_recall':              round(float(np.mean(batch_recalls)), 6),
        'min_batch_accuracy':            round(float(np.min(batch_accuracies)), 6),
        'std_batch_accuracy':            round(float(np.std(batch_accuracies)), 6),
        # efficiency
        'accuracy_per_retrain':          round(avg_acc / retrain_count_safe, 6),
        # retraining cost
        'num_retraining_events':         retrain_count,
        'total_retraining_data':         total_retrain_data,
        'avg_retraining_data_per_event': round(float(np.mean(retrain_data_per_event)) if retrain_data_per_event else 0.0, 2),
        # robustness
        'avg_recovery_gain':             round(float(np.mean(recovery_gains)) if recovery_gains else 0.0, 6),
        'avg_recovery_time':             round(float(np.mean(recovery_times)) if recovery_times else 0.0, 4),
        'fraction_time_below_0_95':      round(below_095_count / n_stream, 6),
        'fraction_time_degraded':        round(degraded_count / n_stream, 6),
        # trigger analysis
        'false_drift_rate':              round(false_drift_count / n_stream, 6),
        'missed_drift_rate':             round(missed_drift_count / n_stream, 6),
        'fraction_trigger_performance':  round(perf_trigs / total_triggers, 6),
        'fraction_trigger_psi':          round(psi_trigs / total_triggers, 6),
    }

    return batchwise_records, overall_record


print("✅ Simulation engine defined.")

✅ Simulation engine defined.


---
## Cell 8 — Simulation Matrix

Defines all 60 simulation runs as a list of tuples:
`(sim_no, dataset, model, strategy, label_available)`.

| Sim range | Dataset | Label availability |
|-----------|---------|--------------------|
| 1–10 | Electricity | Available (performance trigger) |
| 11–20 | Electricity | Delayed (PSI trigger) |
| 21–30 | Gradual concept | Available |
| 31–40 | Gradual concept | Delayed |
| 41–50 | Abrupt concept | Available |
| 51–60 | Abrupt concept | Delayed |

Within each group of 10: strategies cycle through
no_retraining, continuous, cumulative, sliding_window, exponential_decay
for Logistic Regression (runs 1–5) and XGBoost (runs 6–10).

In [10]:
simulation_matrix = [
    # SIM 1–10: ELECTRICITY — AVAILABLE
    (1,  'electricity', 'logistic_regression', 'no_retraining',    True),
    (2,  'electricity', 'logistic_regression', 'continuous',        True),
    (3,  'electricity', 'logistic_regression', 'cumulative',        True),
    (4,  'electricity', 'logistic_regression', 'sliding_window',    True),
    (5,  'electricity', 'logistic_regression', 'exponential_decay', True),
    (6,  'electricity', 'xgboost',             'no_retraining',     True),
    (7,  'electricity', 'xgboost',             'continuous',         True),
    (8,  'electricity', 'xgboost',             'cumulative',         True),
    (9,  'electricity', 'xgboost',             'sliding_window',     True),
    (10, 'electricity', 'xgboost',             'exponential_decay',  True),
    # SIM 11–20: ELECTRICITY — DELAYED
    (11, 'electricity', 'logistic_regression', 'no_retraining',    False),
    (12, 'electricity', 'logistic_regression', 'continuous',        False),
    (13, 'electricity', 'logistic_regression', 'cumulative',        False),
    (14, 'electricity', 'logistic_regression', 'sliding_window',    False),
    (15, 'electricity', 'logistic_regression', 'exponential_decay', False),
    (16, 'electricity', 'xgboost',             'no_retraining',     False),
    (17, 'electricity', 'xgboost',             'continuous',         False),
    (18, 'electricity', 'xgboost',             'cumulative',         False),
    (19, 'electricity', 'xgboost',             'sliding_window',     False),
    (20, 'electricity', 'xgboost',             'exponential_decay',  False),
    # SIM 21–30: GRADUAL + CONCEPT — DELAYED
    (21, 'electricity_gradual_concept', 'logistic_regression', 'no_retraining',    False),
    (22, 'electricity_gradual_concept', 'logistic_regression', 'continuous',        False),
    (23, 'electricity_gradual_concept', 'logistic_regression', 'cumulative',        False),
    (24, 'electricity_gradual_concept', 'logistic_regression', 'sliding_window',    False),
    (25, 'electricity_gradual_concept', 'logistic_regression', 'exponential_decay', False),
    (26, 'electricity_gradual_concept', 'xgboost',             'no_retraining',     False),
    (27, 'electricity_gradual_concept', 'xgboost',             'continuous',         False),
    (28, 'electricity_gradual_concept', 'xgboost',             'cumulative',         False),
    (29, 'electricity_gradual_concept', 'xgboost',             'sliding_window',     False),
    (30, 'electricity_gradual_concept', 'xgboost',             'exponential_decay',  False),
    # SIM 31–40: GRADUAL + CONCEPT — AVAILABLE
    (31, 'electricity_gradual_concept', 'logistic_regression', 'no_retraining',     True),
    (32, 'electricity_gradual_concept', 'logistic_regression', 'continuous',         True),
    (33, 'electricity_gradual_concept', 'logistic_regression', 'cumulative',         True),
    (34, 'electricity_gradual_concept', 'logistic_regression', 'sliding_window',     True),
    (35, 'electricity_gradual_concept', 'logistic_regression', 'exponential_decay',  True),
    (36, 'electricity_gradual_concept', 'xgboost',             'no_retraining',      True),
    (37, 'electricity_gradual_concept', 'xgboost',             'continuous',          True),
    (38, 'electricity_gradual_concept', 'xgboost',             'cumulative',          True),
    (39, 'electricity_gradual_concept', 'xgboost',             'sliding_window',      True),
    (40, 'electricity_gradual_concept', 'xgboost',             'exponential_decay',   True),
    # SIM 41–50: ABRUPT + CONCEPT — DELAYED
    (41, 'electricity_abrupt_concept', 'logistic_regression', 'no_retraining',    False),
    (42, 'electricity_abrupt_concept', 'logistic_regression', 'continuous',        False),
    (43, 'electricity_abrupt_concept', 'logistic_regression', 'cumulative',        False),
    (44, 'electricity_abrupt_concept', 'logistic_regression', 'sliding_window',    False),
    (45, 'electricity_abrupt_concept', 'logistic_regression', 'exponential_decay', False),
    (46, 'electricity_abrupt_concept', 'xgboost',             'no_retraining',     False),
    (47, 'electricity_abrupt_concept', 'xgboost',             'continuous',         False),
    (48, 'electricity_abrupt_concept', 'xgboost',             'cumulative',         False),
    (49, 'electricity_abrupt_concept', 'xgboost',             'sliding_window',     False),
    (50, 'electricity_abrupt_concept', 'xgboost',             'exponential_decay',  False),
    # SIM 51–60: ABRUPT + CONCEPT — AVAILABLE
    (51,  'electricity_abrupt_concept', 'logistic_regression', 'no_retraining',     True),
    (52,  'electricity_abrupt_concept', 'logistic_regression', 'continuous',         True),
    (53,  'electricity_abrupt_concept', 'logistic_regression', 'cumulative',         True),
    (54,  'electricity_abrupt_concept', 'logistic_regression', 'sliding_window',     True),
    (55,  'electricity_abrupt_concept', 'logistic_regression', 'exponential_decay',  True),
    (56,  'electricity_abrupt_concept', 'xgboost',             'no_retraining',      True),
    (57,  'electricity_abrupt_concept', 'xgboost',             'continuous',          True),
    (58,  'electricity_abrupt_concept', 'xgboost',             'cumulative',          True),
    (59,  'electricity_abrupt_concept', 'xgboost',             'sliding_window',      True),
    (60, 'electricity_abrupt_concept', 'xgboost',             'exponential_decay',   True),
]

print(f"✅ Simulation matrix: {len(simulation_matrix)} simulations ready.")

✅ Simulation matrix: 60 simulations ready.


---
## Cell 9 — Execute All 60 Simulations

> **Estimated runtime:** 15–40 minutes depending on Colab hardware allocation.

Runs each of the 60 simulations defined in the matrix above.
Results are accumulated in `batchwise_records` (72 rows × 60 = 4,320 rows)
and `overall_records` (60 rows). Progress is printed after each run.

> **If the session disconnects mid-run:** re-run from Cell 1 and check which
> `sim_no` values are already in `df_overall` before re-executing this cell.

In [11]:
all_batchwise = []
all_overall   = []

n_sims  = len(simulation_matrix)
n_seeds = len(SEEDS)
total_runs = n_sims * n_seeds

print(f"🚀 Starting multi-seed experiment")
print(f"   Simulations : {n_sims}")
print(f"   Seeds       : {SEEDS}")
print(f"   Total runs  : {total_runs}\n")

run_counter = 0
for seed in SEEDS:
    print(f"\n{'='*80}")
    print(f"  SEED {seed}")
    print(f"{'='*80}")

    for (sim_no, dataset_name, model_type, strategy, label_available) in simulation_matrix:
        run_counter += 1
        label_str = 'AVAIL' if label_available else 'DELAY'
        print(
            f"  [{run_counter:>3}/{total_runs}] SIM {sim_no:>3} | "
            f"{dataset_name:35s} | {model_type:20s} | "
            f"{strategy:20s} | {label_str} | seed={seed}",
            end=" ... "
        )

        batchwise_records, overall_record = run_simulation(
            df_input        = datasets[dataset_name],
            dataset_name    = dataset_name,
            model_type      = model_type,
            strategy        = strategy,
            label_available = label_available,
            sim_no          = sim_no,
            seed            = seed,
        )

        all_batchwise.extend(batchwise_records)
        all_overall.append(overall_record)

        print(f"done  (retrains={overall_record['num_retraining_events']:>3},  avg_acc={overall_record['avg_batch_accuracy']:.4f})")

print(f"\n✅ All {total_runs} runs complete  ({n_sims} sims × {n_seeds} seeds)")

🚀 Starting multi-seed experiment
   Simulations : 60
   Seeds       : [42]
   Total runs  : 60


  SEED 42
  [  1/60] SIM   1 | electricity                         | logistic_regression  | no_retraining        | AVAIL | seed=42 ... done  (retrains=  0,  avg_acc=0.7209)
  [  2/60] SIM   2 | electricity                         | logistic_regression  | continuous           | AVAIL | seed=42 ... done  (retrains= 72,  avg_acc=0.7384)
  [  3/60] SIM   3 | electricity                         | logistic_regression  | cumulative           | AVAIL | seed=42 ... done  (retrains= 43,  avg_acc=0.7376)
  [  4/60] SIM   4 | electricity                         | logistic_regression  | sliding_window       | AVAIL | seed=42 ... done  (retrains= 42,  avg_acc=0.7440)
  [  5/60] SIM   5 | electricity                         | logistic_regression  | exponential_decay    | AVAIL | seed=42 ... done  (retrains= 38,  avg_acc=0.7468)
  [  6/60] SIM   6 | electricity                         | xgboost            

---
## Cell 10 — Export Results

Saves both result DataFrames to CSV and Excel format:
- `results_overall.csv/.xlsx` — one row per simulation (60 rows),
  used for strategy comparison tables in Chapter 4
- `results_batchwise.csv/.xlsx` — one row per batch per simulation (4,320 rows),
  used for time-series accuracy plots and per-batch analysis

In [16]:
df_batchwise = pd.DataFrame(all_batchwise)
df_overall   = pd.DataFrame(all_overall)

# ── Per-seed results (full granularity — used for statistical tests) ──
batchwise_path = os.path.join(OUTPUT_PATH, 'results_batchwise.csv')
overall_path   = os.path.join(OUTPUT_PATH, 'results_overall.csv')

df_batchwise.to_csv(batchwise_path, index=False)
df_overall.to_csv(overall_path, index=False)

print(f"✅ Batchwise results  → {batchwise_path}")
print(f"   Shape : {df_batchwise.shape[0]:,} rows × {df_batchwise.shape[1]} columns")
print()
print(f"✅ Overall results    → {overall_path}")
print(f"   Shape : {df_overall.shape[0]} rows × {df_overall.shape[1]} columns")
print()

# ── Aggregated across seeds (mean ± std per simulation config) ────────
NUMERIC_COLS = [
    'avg_batch_accuracy', 'avg_batch_auc',
    'avg_batch_f1', 'avg_batch_precision', 'avg_batch_recall',
    'min_batch_accuracy', 'std_batch_accuracy',
    'accuracy_per_retrain', 'num_retraining_events', 'total_retraining_data',
    'avg_retraining_data_per_event', 'avg_recovery_gain', 'avg_recovery_time',
    'fraction_time_below_0_95', 'fraction_time_degraded',
    'false_drift_rate', 'missed_drift_rate',
    'fraction_trigger_performance', 'fraction_trigger_psi',
]
GROUP_KEYS = ['sim_no', 'dataset', 'model', 'strategy', 'label_available']

agg_mean = df_overall.groupby(GROUP_KEYS)[NUMERIC_COLS].mean().round(6)
agg_std  = df_overall.groupby(GROUP_KEYS)[NUMERIC_COLS].std().round(6)
agg_std.columns  = [c + '_std'  for c in agg_std.columns]

df_overall_agg = pd.concat([agg_mean, agg_std], axis=1).reset_index()
agg_path = os.path.join(OUTPUT_PATH, 'results_overall_aggregated.csv')
df_overall_agg.to_csv(agg_path, index=False)

print(f"✅ Aggregated results → {agg_path}")
print(f"   Shape : {df_overall_agg.shape[0]} rows × {df_overall_agg.shape[1]} columns")
print(f"   (mean ± std across {len(SEEDS)} seeds per simulation config)")

✅ Batchwise results  → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/results_batchwise.csv
   Shape : 4,320 rows × 37 columns

✅ Overall results    → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/results_overall.csv
   Shape : 60 rows × 25 columns

✅ Aggregated results → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/results_overall_aggregated.csv
   Shape : 60 rows × 43 columns
   (mean ± std across 1 seeds per simulation config)


---
## Cell 12 — Descriptive Pairwise Strategy Comparison

Computes pairwise accuracy differences between all strategy combinations
within each (dataset, model, label_available) group.

> **Note on statistical testing:** This experiment uses a single random seed (42),
> which produces one observation per strategy per group. Inferential statistical
> tests (e.g., Wilcoxon signed-rank) require multiple independent observations
> and are therefore not applied here. The pairwise comparison below reports
> observed accuracy differences only, without p-values.

Results are saved to `results_strategy_comparison.csv`.

In [17]:
# from scipy import stats
# from itertools import combinations

# # ── Statistical test function ────────────────────────────────────────
# def pairwise_strategy_test(df_overall, metric='avg_batch_accuracy', alpha=0.05):
#     """
#     Perform pairwise Wilcoxon signed-rank tests between strategies,
#     within each (dataset, model, label_available) scenario group.

#     Uses per-seed values — one observation per seed per strategy.
#     Returns a DataFrame of test results.
#     """
#     STRATEGIES = ['no_retraining', 'continuous', 'cumulative',
#                   'sliding_window', 'exponential_decay']
#     results = []

#     groups = df_overall.groupby(['dataset', 'model', 'label_available'])
#     for (dataset, model, label_avail), grp in groups:
#         strat_present = [s for s in STRATEGIES if s in grp['strategy'].values]
#         for s1, s2 in combinations(strat_present, 2):
#             vals1 = grp[grp['strategy'] == s1].sort_values('seed')[metric].values
#             vals2 = grp[grp['strategy'] == s2].sort_values('seed')[metric].values

#             if len(vals1) < 2 or len(vals2) < 2 or len(vals1) != len(vals2):
#                 continue

#             # Wilcoxon signed-rank (paired, non-parametric)
#             # Falls back to t-test note if all differences are zero
#             diff = vals1 - vals2
#             if np.all(diff == 0):
#                 stat, p_val, test_used = np.nan, 1.0, 'zero_diff'
#             else:
#                 try:
#                     stat, p_val = stats.wilcoxon(vals1, vals2, alternative='two-sided')
#                     test_used = 'wilcoxon'
#                 except Exception:
#                     stat, p_val = stats.ttest_rel(vals1, vals2)
#                     test_used = 'paired_ttest'

#             results.append({
#                 'dataset'        : dataset,
#                 'model'          : model,
#                 'label_available': label_avail,
#                 'strategy_a'     : s1,
#                 'strategy_b'     : s2,
#                 'metric'         : metric,
#                 'mean_a'         : round(float(np.mean(vals1)), 6),
#                 'mean_b'         : round(float(np.mean(vals2)), 6),
#                 'mean_diff'      : round(float(np.mean(vals1 - vals2)), 6),
#                 'statistic'      : round(float(stat), 4) if not np.isnan(stat) else None,
#                 'p_value'        : round(float(p_val), 6),
#                 'significant'    : bool(p_val < alpha),
#                 'test_used'      : test_used,
#                 'winner'         : s1 if np.mean(vals1) > np.mean(vals2) else s2,
#             })

#     return pd.DataFrame(results)


# # ── Run tests for key metrics ────────────────────────────────────────
# METRICS_TO_TEST = [
#     'avg_batch_accuracy',
#     'fraction_time_degraded',
#     'num_retraining_events',
#     'avg_recovery_time',
# ]

# stat_results = {}
# for metric in METRICS_TO_TEST:
#     print(f"Running pairwise tests for: {metric} ...")
#     stat_results[metric] = pairwise_strategy_test(df_overall, metric=metric)

# # Combine all metrics into one DataFrame
# df_stats = pd.concat(stat_results.values(), ignore_index=True)
# stats_path = os.path.join(OUTPUT_PATH, 'results_statistical_tests.csv')
# df_stats.to_csv(stats_path, index=False)
# print(f"\n✅ Statistical test results → {stats_path}")
# print(f"   Shape : {df_stats.shape[0]} rows × {df_stats.shape[1]} columns")

# # ── Quick summary: significant pairs per metric ──────────────────────
# print("\n── Significant pairs summary (p < 0.05) ──")
# summary = (df_stats[df_stats['significant']]
#            .groupby(['metric', 'dataset'])
#            .size()
#            .reset_index(name='n_significant_pairs'))
# print(summary.to_string(index=False) if len(summary) > 0 else "  None found.")

# # ── Show top significant results for accuracy ────────────────────────
# print("\n── Top significant accuracy comparisons (avg_batch_accuracy) ──")
# top = (df_stats[(df_stats['metric'] == 'avg_batch_accuracy') & df_stats['significant']]
#        [['dataset', 'model', 'label_available',
#          'strategy_a', 'strategy_b', 'mean_diff', 'p_value', 'winner']]
#        .sort_values('mean_diff', ascending=False, key=abs)
#        .head(20))
# display(top) if len(top) > 0 else print("  None found.")


Running pairwise tests for: avg_batch_accuracy ...
Running pairwise tests for: fraction_time_degraded ...
Running pairwise tests for: num_retraining_events ...
Running pairwise tests for: avg_recovery_time ...

✅ Statistical test results → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/results_statistical_tests.csv
   Shape : 0 rows × 0 columns

── Significant pairs summary (p < 0.05) ──


KeyError: 'significant'

> This cell runs descriptive (not inferential) pairwise comparisons.
> Results show which strategy achieves higher accuracy in each scenario,
> without statistical significance testing.

In [18]:
# ── Descriptive strategy comparison for single-seed experiment ─────────
def pairwise_strategy_comparison(df_overall, metric='avg_batch_accuracy'):
    """
    Pairwise descriptive comparison between strategies,
    within each (dataset, model, label_available) scenario group.

    For single-seed experiments, this reports observed differences only.
    No inferential p-values are computed.
    """
    STRATEGIES = ['no_retraining', 'continuous', 'cumulative',
                  'sliding_window', 'exponential_decay']
    results = []

    groups = df_overall.groupby(['dataset', 'model', 'label_available'])
    for (dataset, model, label_avail), grp in groups:
        strat_present = [s for s in STRATEGIES if s in grp['strategy'].values]

        for s1, s2 in combinations(strat_present, 2):
            vals1 = grp[grp['strategy'] == s1][metric].values
            vals2 = grp[grp['strategy'] == s2][metric].values

            if len(vals1) == 0 or len(vals2) == 0:
                continue

            val1 = float(vals1[0])
            val2 = float(vals2[0])
            diff = val1 - val2

            if diff > 0:
                winner = s1
            elif diff < 0:
                winner = s2
            else:
                winner = 'tie'

            results.append({
                'dataset'        : dataset,
                'model'          : model,
                'label_available': label_avail,
                'strategy_a'     : s1,
                'strategy_b'     : s2,
                'metric'         : metric,
                'value_a'        : round(val1, 6),
                'value_b'        : round(val2, 6),
                'difference'     : round(diff, 6),
                'abs_difference' : round(abs(diff), 6),
                'winner'         : winner,
                'test_used'      : 'descriptive_single_seed',
                'p_value'        : None,
                'significant'    : None,
            })

    return pd.DataFrame(results)

In [19]:
# ── Run descriptive comparisons for key metrics ───────────────────────
METRICS_TO_COMPARE = [
    'avg_batch_accuracy',
    'fraction_time_degraded',
    'num_retraining_events',
    'avg_recovery_time',
]

comparison_results = {}
for metric in METRICS_TO_COMPARE:
    print(f"Running descriptive comparisons for: {metric} ...")
    comparison_results[metric] = pairwise_strategy_comparison(df_overall, metric=metric)

df_stats = pd.concat(comparison_results.values(), ignore_index=True)

stats_path = os.path.join(OUTPUT_PATH, 'results_strategy_comparisons_single_seed.csv')
df_stats.to_csv(stats_path, index=False)

print(f"\n✅ Descriptive comparison results → {stats_path}")
print(f"   Shape : {df_stats.shape[0]} rows × {df_stats.shape[1]} columns")

Running descriptive comparisons for: avg_batch_accuracy ...
Running descriptive comparisons for: fraction_time_degraded ...
Running descriptive comparisons for: num_retraining_events ...
Running descriptive comparisons for: avg_recovery_time ...

✅ Descriptive comparison results → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/results/results_strategy_comparisons_single_seed.csv
   Shape : 480 rows × 14 columns


---
## Cell 11 — Quick Sanity Check

Verifies the exported results against expected values:
- Confirms 60 rows in `results_overall`
- Confirms 4,320 rows in `results_batchwise`
- Checks that all 60 simulation numbers are present
- Prints summary statistics for a quick plausibility check

In [20]:
print("=" * 100)
print("OVERALL RESULTS SUMMARY")
print("=" * 100)
# Preview — first seed only for readability
display(df_overall[df_overall['seed'] == SEEDS[0]][[
    'sim_no', 'dataset', 'model', 'strategy', 'label_available',
    'avg_batch_accuracy', 'num_retraining_events',
    'fraction_time_degraded', 'avg_recovery_time'
]])

print("\n" + "=" * 100)
print("BATCHWISE SAMPLE — SIM 1, first 10 batches")
print("=" * 100)
display(df_batchwise[df_batchwise['sim_no'] == 1][[
    'env_id', 'accuracy', 'psi_mean', 'baseline_accuracy',
    'accuracy_relative', 'is_degraded', 'retrain_triggered',
    'retrain_trigger_type', 'model_version'
]].head(10))

print("\n" + "=" * 100)
print(f"Batchwise columns ({len(df_batchwise.columns)}): {list(df_batchwise.columns)}")
print(f"Overall   columns ({len(df_overall.columns)})  : {list(df_overall.columns)}")

OVERALL RESULTS SUMMARY


,sim_no,dataset,model,strategy,label_available,avg_batch_accuracy,num_retraining_events,fraction_time_degraded,avg_recovery_time
0,1,electricity,logistic_regression,no_retraining,True,0.720944,0,0.583333,3.2500
1,2,electricity,logistic_regression,continuous,True,0.738361,72,0.666667,2.2941
2,3,electricity,logistic_regression,cumulative,True,0.737556,43,0.597222,3.3333
3,4,electricity,logistic_regression,sliding_window,True,0.744000,42,0.583333,2.3333
4,5,electricity,logistic_regression,exponential_decay,True,0.746833,38,0.527778,2.0625
5,6,electricity,xgboost,no_retraining,True,0.725250,0,0.638889,2.5333
6,7,electricity,xgboost,continuous,True,0.744167,72,0.791667,1.6957
7,8,electricity,xgboost,cumulative,True,0.736611,51,0.708333,2.2222
8,9,electricity,xgboost,sliding_window,True,0.751556,67,0.930556,1.3929
9,10,electricity,xgboost,exponential_decay,True,0.749333,66,0.916667,1.2333



BATCHWISE SAMPLE — SIM 1, first 10 batches


,env_id,accuracy,psi_mean,baseline_accuracy,accuracy_relative,is_degraded,retrain_triggered,retrain_trigger_type,model_version
0,0,0.814,0.288885,0.81,1.004938,False,False,None,1
1,1,0.836,0.373465,0.81,1.032099,False,False,None,1
2,2,0.870,0.356889,0.81,1.074074,False,False,None,1
3,3,0.830,0.335005,0.81,1.024691,False,False,None,1
4,4,0.778,0.400000,0.81,0.960494,False,False,None,1
5,5,0.708,0.400000,0.81,0.874074,True,False,None,1
6,6,0.826,0.279393,0.81,1.019753,False,False,None,1
7,7,0.806,0.263749,0.81,0.995062,False,False,None,1
8,8,0.882,0.231342,0.81,1.088889,False,False,None,1
9,9,0.904,0.153862,0.81,1.116049,False,False,None,1



Batchwise columns (37): ['sim_no', 'env_id', 'dataset_name', 'model_type', 'memory_strategy', 'label_available', 'seed', 'model_version', 'n_samples', 'positive_rate', 'psi_mean', 'accuracy', 'auc', 'f1', 'precision', 'recall', 'baseline_accuracy', 'accuracy_relative', 'performance_drop', 'recovery_time', 'recovery_gain', 'retrain_triggered', 'retrain_trigger_type', 'retrain_count_so_far', 'retrain_data', 'cumulative_retrain_data', 'batches_since_last_retrain', 'threshold_relative', 'threshold_minimum', 'psi_threshold', 'is_degraded', 'is_below_minimum_threshold', 'is_drift', 'is_false_drift', 'is_missed_drift', 'degradation_start_flag', 'is_recovery_complete']
Overall   columns (25)  : ['sim_no', 'dataset', 'model', 'strategy', 'label_available', 'seed', 'avg_batch_accuracy', 'avg_batch_auc', 'avg_batch_f1', 'avg_batch_precision', 'avg_batch_recall', 'min_batch_accuracy', 'std_batch_accuracy', 'accuracy_per_retrain', 'num_retraining_events', 'total_retraining_data', 'avg_retraining_d